<a href="https://colab.research.google.com/github/swarnkarnitin/TrafficMonitoring/blob/main/detr_Traffic_Videos_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os
import yaml
import json
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import albumentations
from transformers import AutoImageProcessor

# dataset_path is already defined as the unzip_path from previous attempts
# Correct the dataset_path to point to the current unzipped directory
dataset_path = "/content/drive/MyDrive/Traffic_Videos/coco_v3/unzip"

# Load annotations from a sample split (e.g., 'train') to get class names
annotation_path_train = os.path.join(dataset_path, 'train', '_annotations.coco.json')

if not os.path.exists(annotation_path_train):
    alternative_annotation_path_train = os.path.join(dataset_path, 'train', 'annotations', '_annotations.coco.json')
    if os.path.exists(alternative_annotation_path_train):
        annotation_path_train = alternative_annotation_path_train
    else:
        raise FileNotFoundError(f"Annotation file not found at {annotation_path_train} or {alternative_annotation_path_train}")


with open(annotation_path_train, 'r') as f:
    coco_train = json.load(f)

# Load class names from the annotations file
classes = [cat['name'] for cat in coco_train['categories']]
id2label = {cat['id']: cat['name'] for cat in coco_train['categories']}
label2id = {cat['name']: cat['id'] for cat in coco_train['categories']}

print("Classes loaded from annotations:", classes)
print("id2label:", id2label)
print("label2id:", label2id)


# Initialize the image processor
checkpoint = "facebook/detr-resnet-50-dc5"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)
print("\nImage processor initialized.")

# Define a simple transform for now (can be expanded later)
transform = albumentations.Compose(
    [
        albumentations.Resize(480, 480),
        albumentations.HorizontalFlip(p=0.5), # Add some basic augmentation
    ],
    bbox_params=albumentations.BboxParams(format="coco", label_fields=["category_id"]), # Use category_id for COCO format
)
print("Transform defined.")


# Define the custom PyTorch Dataset class
class CocoObjectDetectionDataset(Dataset):
    def __init__(self, root_dir, split, image_processor=None, transform=None):
        self.root_dir = root_dir
        self.split = split # 'train' or 'valid'
        self.image_processor = image_processor
        self.transform = transform

        # Construct paths to images and annotations (now directly in root_dir/split)
        self.image_dir = os.path.join(self.root_dir, split) # Images are directly in the split folder
        # Corrected annotation path based on expected COCO format
        self.annotation_path = os.path.join(self.root_dir, split, '_annotations.coco.json')


        # Load annotations
        if not os.path.exists(self.annotation_path):
             # Check for a common alternative path structure if the first one fails
             alternative_annotation_path = os.path.join(self.root_dir, split, 'annotations', '_annotations.coco.json')
             if os.path.exists(alternative_annotation_path):
                 self.annotation_path = alternative_annotation_path
             else:
                raise FileNotFoundError(f"Annotation file not found at {self.annotation_path} or {alternative_annotation_path}")


        with open(self.annotation_path, 'r') as f:
            self.coco = json.load(f)

        # Load class names from the annotations file (redundant if loaded globally, but good for dataset self-containment if needed)
        # self.classes = [cat['name'] for cat in self.coco['categories']]
        # self.id2label = {cat['id']: cat['name'] for cat in self.coco['categories']}
        # self.label2id = {cat['name']: cat['id'] for cat in self.coco['categories']}

        # print(f"Classes loaded from {self.annotation_path}:", self.classes)


        # Create a mapping from image id to annotations
        self.img_id_to_annotations = {}
        for ann in self.coco['annotations']:
            img_id = ann['image_id']
            if img_id not in self.img_id_to_annotations:
                self.img_id_to_annotations[img_id] = []
            self.img_id_to_annotations[img_id].append(ann)

        # Create a list of images
        self.images = self.coco['images']

        # Create a mapping from image id to image info
        self.img_id_to_info = {img['id']: img for img in self.images}


    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        image_path = os.path.join(self.image_dir, img_info['file_name'])

        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        # Get annotations for this image
        annotations = self.img_id_to_annotations.get(img_id, [])

        # Prepare target in the format expected by the image processor
        # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
        # where each dict is a COCO object annotation: {'category_id': int, 'bbox': List[float], 'area': float, 'iscrowd': int}
        target = {'image_id': img_id, 'annotations': annotations, 'width': width, 'height': height}

        # Apply transform if provided
        if self.transform:
            # Albumentations requires numpy array and expects [x_min, y_min, width, height] for bbox
            image_np = np.array(image)
            bboxes_coco = [ann['bbox'] for ann in annotations]
            categories = [ann['category_id'] for ann in annotations]

            # Albumentations requires label_fields to be present even if empty
            if not categories:
                 categories = [0] * len(bboxes_coco) # Provide dummy category if none exist

            transformed = self.transform(image=image_np, bboxes=bboxes_coco, category_id=categories) # Use category_id as defined in bbox_params

            image = Image.fromarray(transformed['image'])
            transformed_bboxes_coco = transformed['bboxes']
            transformed_categories = transformed['category_id']

            # Update annotations with transformed bboxes and categories
            annotations = [] # Reset annotations list
            for i in range(len(transformed_bboxes_coco)):
                 annotations.append({
                        'category_id': transformed_categories[i],
                        'bbox': list(transformed_bboxes_coco[i]), # COCO format: [x_min, y_min, width, height]
                        'area': transformed_bboxes_coco[i][2] * transformed_bboxes_coco[i][3], # Recalculate area
                        'iscrowd': 0, # Assuming no crowd objects
                        'image_id': img_id # Keep original image_id
                    })
            # Update target with transformed annotations
            target['annotations'] = annotations
            target['width'] = image.size[0]
            target['height'] = image.size[1]


        # Apply image processor
        if self.image_processor:
            # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
            # where each dict is a COCO object annotation.
            # It also handles resizing and normalization.
            # We need to provide the original size for the processor to scale the bboxes correctly.

            # Filter out annotations with invalid bboxes after transformation (width or height <= 0)
            valid_annotations = [ann for ann in target['annotations'] if ann['bbox'][2] > 0 and ann['bbox'][3] > 0]
            target['annotations'] = valid_annotations

            # The image processor expects a list of targets, even for a single image
            processed = self.image_processor(images=image, annotations=[target], return_tensors="pt")

            # The processor returns pixel_values, pixel_mask, and labels.
            # The 'labels' key contains the transformed annotations in a specific format.
            # We will return this structure as is, which is expected by the collate_fn.

            return {
                'pixel_values': processed['pixel_values'].squeeze(0), # Remove batch dimension
                'pixel_mask': processed['pixel_mask'].squeeze(0), # Remove batch dimension
                'labels': processed['labels'][0] if processed['labels'] else {'class_labels': torch.tensor([]), 'boxes': torch.tensor([]), 'area': torch.tensor([]), 'iscrowd': torch.tensor([]), 'image_id': torch.tensor([img_id]), 'orig_size': torch.tensor([height, width]), 'size': torch.tensor([target['height'], target['width']])} # Return empty tensors if no labels

            }

        return image, target


# Instantiate the datasets
train_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='train', image_processor=image_processor, transform=transform)
valid_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='valid', image_processor=image_processor, transform=transform) # Assuming 'valid' split exists


# Verify loading by accessing a few samples
print("\nTrain dataset sample 0:")
sample_train = train_dataset[0]
print(sample_train)

print("\nValid dataset sample 0:")
sample_valid = valid_dataset[0]
print(sample_valid)

Classes loaded from annotations: ['car', 'bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck']
id2label: {0: 'car', 1: 'bus', 2: 'car', 3: 'microbus', 4: 'motorbike', 5: 'pickup-van', 6: 'truck'}
label2id: {'car': 2, 'bus': 1, 'microbus': 3, 'motorbike': 4, 'pickup-van': 5, 'truck': 6}

Image processor initialized.
Transform defined.

Train dataset sample 0:
{'pixel_values': tensor([[[-1.2788, -1.3130, -1.3644,  ..., -1.5185, -1.4843, -1.4672],
         [-1.1589, -1.2103, -1.2788,  ..., -1.5014, -1.4672, -1.4500],
         [-0.9877, -1.0562, -1.1418,  ..., -1.4843, -1.4500, -1.4329],
         ...,
         [-1.3130, -1.3473, -1.3815,  ..., -0.4397, -0.4568, -0.4568],
         [-1.2788, -1.3130, -1.3473,  ..., -0.4568, -0.4739, -0.4739],
         [-1.2617, -1.2959, -1.3302,  ..., -0.4739, -0.4739, -0.4739]],

        [[-1.0728, -1.1078, -1.1429,  ..., -1.4230, -1.3880, -1.3704],
         [-0.9678, -1.0203, -1.0728,  ..., -1.4055, -1.3704, -1.3529],
         [-0.8102, -0.8803, -0

In [12]:
from transformers import AutoModelForObjectDetection

# id2label and label2id are already defined in the previous step based on data_yaml

model = AutoModelForObjectDetection.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

In [13]:
from transformers import TrainingArguments, Trainer
import torch

# Redefine training_args with increased max_steps and epochs
training_args = TrainingArguments(
    output_dir="detr-resnet-50-vehicle-finetuned", # Changed output directory name
    per_device_train_batch_size=1, # Further reduced batch size
    num_train_epochs=10,  # Increased epochs
    max_steps=5000,  # Increased max steps
    fp16=True,
    save_steps=50, # Save checkpoints more frequently
    logging_steps=50, # Log more frequently
    learning_rate=1e-5,
    weight_decay=1e-4,
    save_total_limit=3, # Keep more checkpoints
    remove_unused_columns=False,
    push_to_hub=False, # Set to False to avoid accidental pushes
)

# Define a collate function to prepare batches for DETR
def collate_fn(batch):
    pixel_values = [item["pixel_values"] for item in batch]
    pixel_mask = [item["pixel_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    # Stack pixel_values and pixel_mask
    pixel_values = torch.stack(pixel_values)
    pixel_mask = torch.stack(pixel_mask)

    # Return the batch in the format expected by the model's forward pass
    # The labels should be a list of dictionaries, where each dictionary
    # corresponds to an image and contains the annotations for that image.
    return {"pixel_values": pixel_values, "pixel_mask": pixel_mask, "labels": labels}


# Ensure the Trainer is instantiated with the correct arguments
# model, train_dataset, collate_fn, and image_processor should be available from previous successful steps

trainer = Trainer(
    model=model,
    args=training_args, # Use the training_args defined here
    data_collator=collate_fn,
    train_dataset=train_dataset,
    # eval_dataset=test_dataset, # Keep evaluation removed for now
    tokenizer=image_processor, # Use the image_processor as the tokenizer
)

# Start the training process
trainer.train()

/tmp/ipython-input-144478659.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nitin_swarnkar_ampba2025s (nitin_swarnkar_ampba2025s-isb) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,2.846900
100,2.699700
150,2.428400
200,2.274500
250,2.167800
300,2.154900
350,2.008200
400,2.017200
450,1.794600
500,1.886900


TrainOutput(global_step=5000, training_loss=1.5876382186889648, metrics={'train_runtime': 4029.8115, 'train_samples_per_second': 1.241, 'train_steps_per_second': 1.241, 'total_flos': 2.389108608e+18, 'train_loss': 1.5876382186889648, 'epoch': 0.7587253414264037})

In [ ]:
import os

# Define the desired save path in Google Drive
save_path = "/content/drive/MyDrive/Traffic_Videos/Trained_Model/detr_model_2"

# Create the directory if it doesn't exist
os.makedirs(save_path, exist_ok=True)

# Save the trained model to the specified path
# The Trainer object is available from the previous steps.
trainer.save_model(save_path)

print(f"Model saved to {save_path}")

In [ ]:
import cv2
import numpy as np
from PIL import Image, ImageDraw
import torch
import os # Import os to use os.path.exists

# Define the input video path
video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# Define the output video path
output_video_path = "/content/drive/MyDrive/Traffic_Videos/detr_output_video_finetuned3.avi"

# Check if the video file exists
if not os.path.exists(video_path):
    print(f"Error: Video file not found at {video_path}")
else:
    # Re-open the video file
    cap = cv2.VideoCapture(video_path)

    # Check if the video was opened successfully
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
    else:
        # Get video properties
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))

        # Define the codec and create VideoWriter object
        fourcc = cv2.VideoWriter_fourcc(*'XVID') # You can use other codecs like 'MP4V'
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

        # Ensure the model is in evaluation mode
        # model is defined in a previous cell
        model.eval()

        # Define a threshold for detections
        detection_threshold = 0.5 # Adjusted detection threshold

        # Define device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        # Iterate through the video frames
        frame_count = 0
        # max_frames_to_process = 100  # Commented out the limit to process the whole video

        while cap.isOpened(): # Process all frames
            ret, frame = cap.read()
            if not ret:
                break

            # Convert the OpenCV BGR image to RGB for the model
            image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            # Prepare the image for the model
            # image_processor is defined in a previous cell
            inputs = image_processor(images=image, return_tensors="pt").to(device) # device is defined in a previous cell

            # Perform inference
            with torch.no_grad():
                outputs = model(**inputs)

            # Post-process the model outputs
            target_sizes = torch.tensor([image.size[::-1]]).to(device)
            results = image_processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=detection_threshold)[0]

            # Add print statements to inspect the results
            print(f"Frame {frame_count}: Number of detections above threshold ({detection_threshold}): {len(results['scores'])}")
            if len(results['scores']) > 0:
                print("Scores:", results['scores'])
                print("Labels:", results['labels'])
                print("Boxes:", results['boxes'])


            # Draw bounding boxes and labels on the frame
            draw = ImageDraw.Draw(image)
            for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
                # Convert box from [x_min, y_min, x_max, y_max]
                x1, y1, x2, y2 = box.tolist()
                draw.rectangle((x1, y1, x2, y2), outline="red", width=2)
                # Ensure label is an integer before accessing id2label (id2label is defined in a previous cell)
                label_text = f"{id2label[int(label)]}: {score:.2f}"
                # Add text slightly above the bounding box
                text_position = (x1, y1 - 10) if y1 - 10 > 0 else (x1, y1 + 5)
                draw.text(text_position, label_text, fill="red")
            print(f"Frame {frame_count}: Drawing complete.")


            # Convert the PIL image back to OpenCV format (BGR)
            frame_with_detections = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            print(f"Frame {frame_count}: Converted to OpenCV format.")

            # Write the frame with detections to the output video
            out.write(frame_with_detections)
            print(f"Frame {frame_count}: Written to output video.")

            frame_count += 1
            if frame_count % 100 == 0:  # Print progress every 100 frames
                print(f"Processed {frame_count} frames.")


        # Release everything when job is finished
        cap.release()
        out.release()

        print(f"Object detection complete. Output video saved to {output_video_path}")

In [ ]:
# import os
# import yaml

# dataset_path = "/content/drive/MyDrive/Traffic_Videos/roboflow_images/unzipped_vehicle_data"

# # Look for a classes.txt file
# classes_file = os.path.join(dataset_path, 'classes.txt')
# if os.path.exists(classes_file):
#     print(f"\nContent of classes.txt:")
#     with open(classes_file, 'r') as f:
#         print(f.read())
# else:
#     print("\nclasses.txt not found directly in the dataset path. Checking for data.yaml...")
#     data_yaml_path = os.path.join(dataset_path, 'data.yaml')
#     if os.path.exists(data_yaml_path):
#         print(f"\nContent of data.yaml:")
#         with open(data_yaml_path, 'r') as f:
#             data_yaml = yaml.safe_load(f)
#             print(data_yaml)
#             if 'names' in data_yaml:
#                 print("\nClass names found in data.yaml:")
#                 print(data_yaml['names'])
#             else:
#                 print("\n'names' field not found in data.yaml.")
#     else:
#         print("\ndata.yaml not found either.")


classes.txt not found directly in the dataset path. Checking for data.yaml...

Content of data.yaml:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck'], 'roboflow': {'workspace': 'test-k3dls', 'project': 'vehicle-9x7y5', 'version': 3, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/test-k3dls/vehicle-9x7y5/dataset/3'}}

Class names found in data.yaml:
['bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck']


Update the notebook code to load your custom dataset from the specified Google Drive path.

Define a custom PyTorch Dataset class to load the images and parse the YOLO annotations from the specified Google Drive path for both training and testing sets.

In [3]:
# import os
# import glob
# from PIL import Image
# import torch
# from torch.utils.data import Dataset
# import numpy as np
# import albumentations
# import yaml
# from transformers import AutoImageProcessor

# # Define the dataset path
# dataset_path = "/content/drive/MyDrive/Traffic_Videos/roboflow_images/unzipped_vehicle_data"

# # Load class names from data.yaml
# data_yaml_path = os.path.join(dataset_path, 'data.yaml')
# with open(data_yaml_path, 'r') as f:
#     data_yaml = yaml.safe_load(f)
#     classes = data_yaml['names']
#     id2label = {i: name for i, name in enumerate(classes)}
#     label2id = {name: i for i, name in enumerate(classes)}


# # Initialize the image processor and transform
# checkpoint = "facebook/detr-resnet-50-dc5"
# image_processor = AutoImageProcessor.from_pretrained(checkpoint)

# transform = albumentations.Compose(
#     [
#         albumentations.Resize(480, 480),
#         albumentations.HorizontalFlip(p=1.0),
#         albumentations.RandomBrightnessContrast(p=1.0),
#     ],
#     bbox_params=albumentations.BboxParams(format="coco", label_fields=["category"]),
# )


# class YoloObjectDetectionDataset(Dataset):
#     def __init__(self, root_dir, split, image_processor=None, transform=None):
#         self.root_dir = root_dir
#         self.split = split
#         self.image_processor = image_processor
#         self.transform = transform  # Assign the transform here
#         self.image_files = glob.glob(os.path.join(self.root_dir, self.split, 'images', '*.jpg'))
#         self.image_files.extend(glob.glob(os.path.join(self.root_dir, self.split, 'images', '*.jpeg')))
#         self.image_files.extend(glob.glob(os.path.join(self.root_dir, self.split, 'images', '*.png')))
#         self.image_files.sort() # Ensure consistent order

#         self.label_files = [p.replace('images', 'labels').replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt') for p in self.image_files]

#         # Load class names from data.yaml
#         data_yaml_path = os.path.join(self.root_dir, 'data.yaml')
#         with open(data_yaml_path, 'r') as f:
#             data_yaml = yaml.safe_load(f)
#             self.classes = data_yaml['names']
#         self.id2label = {i: name for i, name in enumerate(self.classes)}
#         self.label2id = {name: i for i, name in enumerate(self.classes)}

#     def __len__(self):
#         return len(self.image_files)

#     def __getitem__(self, idx):
#         image_path = self.image_files[idx]
#         label_path = self.label_files[idx]

#         image = Image.open(image_path).convert("RGB")
#         width, height = image.size

#         annotations = []
#         if os.path.exists(label_path):
#             with open(label_path, 'r') as f:
#                 for line in f.readlines():
#                     class_id, x_center, y_center, w, h = map(float, line.split())

#                     # Convert normalized YOLO format to absolute pixel coordinates (x_min, y_min, x_max, y_max)
#                     x_min = (x_center - w / 2) * width
#                     y_min = (y_center - h / 2) * height
#                     x_max = (x_center + w / 2) * width
#                     y_max = (y_center + h / 2) * height

#                     annotations.append({
#                         'category_id': int(class_id),
#                         'bbox': [x_min, y_min, x_max - x_min, y_max - y_min], # COCO format: [x_min, y_min, width, height]
#                         'area': (x_max - x_min) * (y_max - y_min),
#                         'iscrowd': 0, # Assuming no crowd objects
#                         'image_id': idx # Use index as image_id for simplicity
#                     })


#         target = {'image_id': idx, 'annotations': annotations, 'width': width, 'height': height}

#         if self.transform:
#              # Albumentations requires numpy array and expects [x_min, y_min, width, height] for bbox
#             image_np = np.array(image)
#             bboxes_coco = [ann['bbox'] for ann in annotations]
#             categories = [ann['category_id'] for ann in annotations]

#             # Pass data to transform in a dictionary with keys matching label_fields
#             transformed = self.transform(image=image_np, bboxes=bboxes_coco, category=categories) # Always pass category

#             image = Image.fromarray(transformed['image'])
#             transformed_bboxes_coco = transformed['bboxes']
#             transformed_categories = transformed['category']

#             # Update annotations with transformed bboxes and categories
#             annotations = [] # Reset annotations list
#             for i in range(len(transformed_bboxes_coco)):
#                  annotations.append({
#                         'category_id': transformed_categories[i],
#                         'bbox': list(transformed_bboxes_coco[i]), # COCO format: [x_min, y_min, width, height]
#                         'area': transformed_bboxes_coco[i][2] * transformed_bboxes_coco[i][3], # Recalculate area
#                         'iscrowd': 0, # Assuming no crowd objects
#                         'image_id': idx # Use index as image_id for simplicity
#                     })


#         if self.image_processor:
#             # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#             # where each dict is a COCO object annotation.
#             # It also handles resizing and normalization.
#             # We need to provide the original size for the processor to scale the bboxes correctly.

#             # Filter out annotations with invalid bboxes after transformation
#             valid_annotations = [ann for ann in annotations if ann['bbox'][2] > 0 and ann['bbox'][3] > 0]

#             # Restructure annotations to match the expected COCO format for the processor
#             processor_annotations = [{'image_id': idx, 'annotations': valid_annotations}]

#             processed = self.image_processor(images=[image], annotations=processor_annotations, return_tensors="pt")

#             # The processor returns pixel_values, pixel_mask, and labels.
#             # The 'labels' key contains the transformed annotations in a specific format.
#             # We need to extract the relevant parts and ensure they are in the correct structure
#             # for the DETR model's loss calculation.

#             # The image processor returns a dictionary with keys 'pixel_values', 'pixel_mask', and 'labels'.
#             # The 'labels' value is a list of dictionaries, one for each image in the batch.
#             # For a single image, the 'labels' list contains one dictionary with keys like 'class_labels', 'boxes', etc.
#             # We will return this structure as is, which is expected by the collate_fn.

#             return {
#                 'pixel_values': processed['pixel_values'].squeeze(0), # Remove batch dimension
#                 'pixel_mask': processed['pixel_mask'].squeeze(0), # Remove batch dimension
#                 'labels': processed['labels'][0] if processed['labels'] else {'class_labels': torch.tensor([]), 'boxes': torch.tensor([]), 'area': torch.tensor([]), 'iscrowd': torch.tensor([]), 'image_id': torch.tensor([idx]), 'orig_size': torch.tensor([height, width]), 'size': torch.tensor([image.size[1], image.size[0]])} # Return empty tensors if no labels

#             }

#         return image, target


# # Instantiate the datasets
# train_dataset = YoloObjectDetectionDataset(root_dir=dataset_path, split='train', image_processor=image_processor, transform=transform)
# test_dataset = YoloObjectDetectionDataset(root_dir=dataset_path, split='valid', image_processor=image_processor, transform=transform)

# # Verify loading by accessing a few samples
# print("Train dataset sample 0:")
# sample_train = train_dataset[0]
# print(sample_train)

# print("\nTest dataset sample 0:")
# sample_test = test_dataset[0]
# print(sample_test)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


Train dataset sample 0:
{'pixel_values': tensor([[[2.2489, 2.2489, 2.2489,  ..., 2.1119, 2.1119, 2.1119],
         [2.2489, 2.2489, 2.2489,  ..., 2.1119, 2.1119, 2.1119],
         [2.2489, 2.2489, 2.2489,  ..., 2.1119, 2.1119, 2.1119],
         ...,
         [1.5982, 1.5982, 1.5982,  ..., 1.3242, 1.3242, 1.3242],
         [1.6153, 1.6153, 1.5982,  ..., 1.4612, 1.4440, 1.4269],
         [1.6153, 1.6153, 1.5982,  ..., 1.5639, 1.5297, 1.4954]],

        [[2.4286, 2.4286, 2.4286,  ..., 2.3585, 2.3585, 2.3585],
         [2.4286, 2.4286, 2.4286,  ..., 2.3585, 2.3585, 2.3585],
         [2.4286, 2.4286, 2.4286,  ..., 2.3585, 2.3585, 2.3585],
         ...,
         [1.6758, 1.6758, 1.6758,  ..., 1.4132, 1.4132, 1.4132],
         [1.6933, 1.6933, 1.6758,  ..., 1.5707, 1.5357, 1.5007],
         [1.6933, 1.6933, 1.6758,  ..., 1.6758, 1.6057, 1.5707]],

        [[2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
         [2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
         [2.6400,

Update the model configuration to reflect the classes in your custom dataset.

Define the id2label and label2id dictionaries based on the classes loaded from the data.yaml file and then load the AutoModelForObjectDetection using these dictionaries and the specified checkpoint, ensuring ignore_mismatched_sizes is set to True.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

Run the training process with the modified data loading and preprocessing.

Instantiate the Trainer and start the training process with the custom dataset and model.

In [5]:
# from transformers import TrainingArguments, Trainer
# import torch

# # Redefine training_args with a further reduced batch size
# training_args = TrainingArguments(
#     output_dir="detr-resnet-50-vehicle-finetuned", # Changed output directory name
#     per_device_train_batch_size=1, # Further reduced batch size
#     num_train_epochs=20,
#     max_steps=1000,
#     fp16=True,
#     save_steps=10,
#     logging_steps=30,
#     learning_rate=1e-5,
#     weight_decay=1e-4,
#     save_total_limit=2,
#     remove_unused_columns=False,
#     push_to_hub=False, # Set to False to avoid accidental pushes
# )

# # Define a collate function to prepare batches for DETR
# def collate_fn(batch):
#     pixel_values = [item["pixel_values"] for item in batch]
#     pixel_mask = [item["pixel_mask"] for item in batch]
#     labels = [item["labels"] for item in batch]

#     # Stack pixel_values and pixel_mask
#     pixel_values = torch.stack(pixel_values)
#     pixel_mask = torch.stack(pixel_mask)

#     # Return the batch in the format expected by the model's forward pass
#     # The labels should be a list of dictionaries, where each dictionary
#     # corresponds to an image and contains the annotations for that image.
#     return {"pixel_values": pixel_values, "pixel_mask": pixel_mask, "labels": labels}


# # Ensure the Trainer is instantiated with the correct arguments
# # model, train_dataset, collate_fn, and image_processor should be available from previous successful steps

# trainer = Trainer(
#     model=model,
#     args=training_args, # Use the training_args defined here
#     data_collator=collate_fn,
#     train_dataset=train_dataset,
#     # eval_dataset=test_dataset, # Keep evaluation removed for now
#     tokenizer=image_processor, # Use the image_processor as the tokenizer
# )

# # Start the training process
# trainer.train()

# # Define the desired save path in Google Drive
# save_path = "/content/drive/MyDrive/Traffic_Videos/Trained_Model/detr_model"

# # Create the directory if it doesn't exist
# os.makedirs(save_path, exist_ok=True)

# # Save the trained model to the specified path
# # The Trainer object is available from the previous steps.
# trainer.save_model(save_path)

# print(f"Model saved to {save_path}")

/tmp/ipython-input-2476839990.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nitin_swarnkar_ampba2025s (nitin_swarnkar_ampba2025s-isb) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
30,1.957000
60,1.240800
90,0.612600
120,0.233600
150,0.089800
180,0.043500
210,0.029100
240,0.023100
270,0.019700
300,0.017200


Model saved to /content/drive/MyDrive/Traffic_Videos/Trained_Model/detr_model


In [ ]:
# from transformers import AutoModelForObjectDetection
# import torch

# # Define the path to the saved model in your Google Drive
# output_dir = "/content/drive/MyDrive/Traffic_Videos/detr-resnet-50-vehicle-finetuned_v2"

# # Load the model
# # We need to make sure the id2label and label2id are defined, they were defined in a previous cell
# # If the kernel was reset, redefine them
# if 'id2label' not in globals() or 'label2id' not in globals():
#     # Assuming data.yaml was loaded previously and classes are available
#     # If not, you might need to re-run the cell that loads data.yaml
#     data_yaml_path = "/content/drive/MyDrive/Traffic_Videos/roboflow_images/unzipped_vehicle_data/data.yaml"
#     import yaml
#     with open(data_yaml_path, 'r') as f:
#         data_yaml = yaml.safe_load(f)
#         classes = data_yaml['names']
#         id2label = {i: name for i, name in enumerate(classes)}
#         label2id = {name: i for i, name in enumerate(classes)}


# model = AutoModelForObjectDetection.from_pretrained(output_dir, id2label=id2label, label2id=label2id)

# # Move model to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# print(f"Model loaded from {output_dir} and moved to {device}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

Model loaded from /content/drive/MyDrive/Traffic_Videos/detr-resnet-50-vehicle-finetuned_v2 and moved to cuda


In [1]:
# import cv2
# import os

# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# # Check if the video file exists
# if not os.path.exists(video_path):
#     print(f"Error: Video file not found at {video_path}")
# else:
#     # Open the video file
#     cap = cv2.VideoCapture(video_path)

#     # Check if the video was opened successfully
#     if not cap.isOpened():
#         print(f"Error: Could not open video file {video_path}")
#     else:
#         # Get video properties
#         frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#         fps = int(cap.get(cv2.CAP_PROP_FPS))
#         frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

#         print(f"Video loaded successfully:")
#         print(f"Frame count: {frame_count}")
#         print(f"FPS: {fps}")
#         print(f"Frame width: {frame_width}")
#         print(f"Frame height: {frame_height}")

#         # Release the video capture object
#         cap.release()

Error: Video file not found at /content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV


In [ ]:
# import cv2
# import numpy as np
# from PIL import Image, ImageDraw
# import torch

# # Define the output video path
# output_video_path = "/content/drive/MyDrive/Traffic_Videos/detr_output_video3.avi"

# # Re-open the video file
# cap = cv2.VideoCapture(video_path)

# # Get video properties
# frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# fps = int(cap.get(cv2.CAP_PROP_FPS))

# # Define the codec and create VideoWriter object
# fourcc = cv2.VideoWriter_fourcc(*'XVID') # You can use other codecs like 'MP4V'
# out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# # Ensure the model is in evaluation mode
# model.eval()

# # Define a threshold for detections
# detection_threshold = 0.3 # Lowered the detection threshold

# # Iterate through the video frames
# frame_count = 0
# max_frames_to_process = 100  # Set the maximum number of frames to process

# while cap.isOpened() and frame_count < max_frames_to_process:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # Convert the OpenCV BGR image to RGB for the model
#     image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

#     # Prepare the image for the model
#     # Assuming image_processor is defined from a previous cell
#     inputs = image_processor(images=image, return_tensors="pt").to(device)

#     # Perform inference
#     with torch.no_grad():
#         outputs = model(**inputs)

#     # Process the model outputs
#     # The outputs contain predicted bounding boxes, class labels, and confidence scores
#     # We need to post-process these outputs to get the actual detections
#     # You might need to refer to the DETR documentation or examples for post-processing details.
#     # A common approach involves applying non-maximum suppression and filtering by confidence score.

#     # A more standard way using the image_processor's post-processing
#     target_sizes = torch.tensor([image.size[::-1]]).to(device)
#     results = image_processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=detection_threshold)[0]

#     # Add print statements to inspect the results
#     print(f"Frame {frame_count}: Number of detections above threshold ({detection_threshold}): {len(results['scores'])}")
#     if len(results['scores']) > 0:
#         print("Scores:", results['scores'])
#         print("Labels:", results['labels'])
#         print("Boxes:", results['boxes'])


#     # Draw bounding boxes and labels on the frame
#     draw = ImageDraw.Draw(image)
#     for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
#         # Convert box from [x_min, y_min, x_max, y_max]
#         x1, y1, x2, y2 = box.tolist()
#         draw.rectangle((x1, y1, x2, y2), outline="red", width=2)
#         # Ensure label is an integer before accessing id2label
#         label_text = f"{model.config.id2label[int(label)]}: {score:.2f}"
#         draw.text((x1, y1), label_text, fill="white")


#     # Convert the PIL image back to OpenCV format (BGR)
#     frame_with_detections = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

#     # Write the frame with detections to the output video
#     out.write(frame_with_detections)

#     frame_count += 1
#     if frame_count % 10 == 0:  # Print progress every 10 frames
#         print(f"Processed {frame_count} frames.")


# # Release everything when job is finished
# cap.release()
# out.release()
# # cv2.destroyAllWindows() # Removed this line

# print(f"Object detection complete. Output video saved to {output_video_path}")

Frame 0: Number of detections above threshold (0.7): 0
Frame 1: Number of detections above threshold (0.7): 0
Frame 2: Number of detections above threshold (0.7): 0
Frame 3: Number of detections above threshold (0.7): 0
Frame 4: Number of detections above threshold (0.7): 0
Frame 5: Number of detections above threshold (0.7): 0
Frame 6: Number of detections above threshold (0.7): 0
Frame 7: Number of detections above threshold (0.7): 0
Frame 8: Number of detections above threshold (0.7): 0
Frame 9: Number of detections above threshold (0.7): 0
Processed 10 frames.
Frame 10: Number of detections above threshold (0.7): 0
Frame 11: Number of detections above threshold (0.7): 0
Frame 12: Number of detections above threshold (0.7): 0
Frame 13: Number of detections above threshold (0.7): 0
Frame 14: Number of detections above threshold (0.7): 0
Frame 15: Number of detections above threshold (0.7): 0
Frame 16: Number of detections above threshold (0.7): 0
Frame 17: Number of detections above 

# Task
Unzip "/content/drive/MyDrive/Traffic_Videos/coco_v3/vehicle.v3i.coco.zip" into "/content/drive/MyDrive/Traffic_Videos/coco_v3/unzip", train a DETR model on the images in "/content/drive/MyDrive/Traffic_Videos/coco_v3/unzip", and then run object detection on the video "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV", saving the output video with bounding boxes and class labels.

## Install necessary libraries

### Subtask:
Install the required libraries for object detection, such as transformers, torch, and albumentations.


**Reasoning**:
The subtask is to install the required libraries. I will use pip to install the specified libraries quietly.



In [ ]:
!pip install --quiet transformers torch albumentations PyYAML

## Prepare the dataset

### Subtask:
Prepare the dataset by unzipping the provided archive, loading images and annotations, and creating a custom PyTorch Dataset.


**Reasoning**:
Unzip the dataset file and then define the dataset path and load the class names from the data.yaml file.



In [ ]:
# import zipfile
# import os
# import yaml

# # Define the paths
# zip_path = "/content/drive/MyDrive/Traffic_Videos/coco_v3/vehicle.v3i.coco.zip"
# unzip_path = "/content/drive/MyDrive/Traffic_Videos/coco_v3/unzip"

# # Create the unzip directory if it doesn't exist
# os.makedirs(unzip_path, exist_ok=True)

# # Unzip the file
# print(f"Unzipping {zip_path} to {unzip_path}...")
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(unzip_path)
# print("Unzipping complete.")

# # Define the dataset path to the unzipped directory
# dataset_path = unzip_path

# # Load class names from data.yaml
# data_yaml_path = os.path.join(dataset_path, 'data.yaml')
# if not os.path.exists(data_yaml_path):
#     # If data.yaml is not directly in the unzip folder, check common subdirectories like 'data' or the dataset name itself
#     # This part is an assumption based on common dataset structures
#     potential_data_yaml_path = os.path.join(dataset_path, 'data', 'data.yaml')
#     if os.path.exists(potential_data_yaml_path):
#         data_yaml_path = potential_data_yaml_path
#     else:
#          potential_data_yaml_path = os.path.join(dataset_path, 'vehicle.v3i.coco', 'data.yaml') # Assuming 'vehicle.v3i.coco' is the top-level folder inside the zip
#          if os.path.exists(potential_data_yaml_path):
#              data_yaml_path = potential_data_yaml_path
#          else:
#             raise FileNotFoundError(f"data.yaml not found in {dataset_path} or common subdirectories.")


# print(f"Loading classes from {data_yaml_path}")
# with open(data_yaml_path, 'r') as f:
#     data_yaml = yaml.safe_load(f)
#     classes = data_yaml['names']
#     id2label = {i: name for i, name in enumerate(classes)}
#     label2id = {name: i for i, name in enumerate(classes)}

# print("Classes loaded:", classes)
# print("id2label:", id2label)
# print("label2id:", label2id)

Unzipping /content/drive/MyDrive/Traffic_Videos/coco_v3/vehicle.v3i.coco.zip to /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip...
Unzipping complete.


FileNotFoundError: data.yaml not found in /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip or common subdirectories.

**Reasoning**:
The previous attempt to find data.yaml failed. I need to list the contents of the unzipped directory to locate the data.yaml file and then proceed with loading it.



In [ ]:
# import os
# import yaml
# import json # Import json for handling annotations

# # The unzip_path and dataset_path are already defined from the previous attempt
# # List the contents of the unzipped directory to find data.yaml
# print(f"Contents of {dataset_path}:")
# print(os.listdir(dataset_path))

# # Assuming data.yaml is in a subdirectory based on the error, let's try a common structure
# # Look for the subdirectory that was likely created by the unzip process
# # Based on the zip file name 'vehicle.v3i.coco.zip', the top-level folder inside might be 'vehicle.v3i.coco'
# potential_dataset_root = os.path.join(dataset_path, 'vehicle.v3i.coco')

# if os.path.exists(potential_dataset_root):
#     print(f"\nContents of {potential_dataset_root}:")
#     print(os.listdir(potential_dataset_root))
#     data_yaml_path = os.path.join(potential_dataset_root, 'data.yaml')
#     if os.path.exists(data_yaml_path):
#         print(f"\nLoading classes from {data_yaml_path}")
#         with open(data_yaml_path, 'r') as f:
#             data_yaml = yaml.safe_load(f)
#             classes = data_yaml['names']
#             id2label = {i: name for i, name in enumerate(classes)}
#             label2id = {name: i for i, name in enumerate(classes)}

#         print("Classes loaded:", classes)
#         print("id2label:", id2label)
#         print("label2id:", label2id)
#     else:
#         raise FileNotFoundError(f"data.yaml not found in {potential_dataset_root}.")
# else:
#     raise FileNotFoundError(f"Could not find the expected subdirectory 'vehicle.v3i.coco' in {dataset_path}.")

# # Initialize the image processor
# from transformers import AutoImageProcessor
# checkpoint = "facebook/detr-resnet-50-dc5"
# image_processor = AutoImageProcessor.from_pretrained(checkpoint)
# print("\nImage processor initialized.")

# # Define a simple transform for now (can be expanded later)
# import albumentations
# transform = albumentations.Compose(
#     [
#         albumentations.Resize(480, 480),
#         albumentations.HorizontalFlip(p=0.5), # Add some basic augmentation
#     ],
#     bbox_params=albumentations.BboxParams(format="coco", label_fields=["category_id"]), # Use category_id for COCO format
# )
# print("Transform defined.")


# # Define the custom PyTorch Dataset class
# import torch
# from torch.utils.data import Dataset
# from PIL import Image
# import numpy as np

# class CocoObjectDetectionDataset(Dataset):
#     def __init__(self, root_dir, split, image_processor=None, transform=None):
#         self.root_dir = root_dir
#         self.split = split # 'train' or 'valid'
#         self.image_processor = image_processor
#         self.transform = transform

#         # Construct paths to images and annotations
#         self.image_dir = os.path.join(self.root_dir, split, 'images')
#         self.annotation_path = os.path.join(self.root_dir, split, '_annotations.coco.json')

#         # Load annotations
#         if not os.path.exists(self.annotation_path):
#              raise FileNotFoundError(f"Annotation file not found at {self.annotation_path}")

#         with open(self.annotation_path, 'r') as f:
#             self.coco = json.load(f)

#         # Create a mapping from image id to annotations
#         self.img_id_to_annotations = {}
#         for ann in self.coco['annotations']:
#             img_id = ann['image_id']
#             if img_id not in self.img_id_to_annotations:
#                 self.img_id_to_annotations[img_id] = []
#             self.img_id_to_annotations[img_id].append(ann)

#         # Create a list of images, filtering out images without annotations if needed (optional)
#         # For COCO format, the 'images' list contains all images, even those without annotations.
#         # We'll keep all images and handle cases with no annotations in __getitem__.
#         self.images = self.coco['images']

#         # Create a mapping from image id to image info
#         self.img_id_to_info = {img['id']: img for img in self.images}


#         # Load class names from data.yaml (redundant if already loaded, but good for self-contained dataset)
#         data_yaml_path = os.path.join(os.path.dirname(self.root_dir), 'data.yaml') # Assuming data.yaml is one level up from train/valid folders
#         if not os.path.exists(data_yaml_path):
#              # Try one level up from the root_dir
#              data_yaml_path = os.path.join(self.root_dir, 'data.yaml')
#              if not os.path.exists(data_yaml_path):
#                  # Try inside the root_dir in a common data folder
#                  data_yaml_path = os.path.join(self.root_dir, 'data', 'data.yaml')
#                  if not os.path.exists(data_yaml_path):
#                      # Try inside the root_dir in a folder named after the dataset
#                      data_yaml_path = os.path.join(self.root_dir, 'vehicle.v3i.coco', 'data.yaml')
#                      if not os.path.exists(data_yaml_path):
#                          print(f"Warning: data.yaml not found in expected locations relative to {self.root_dir}. Using globally defined classes.")
#                          # Fallback to globally defined classes if available
#                          if 'classes' in globals():
#                              self.classes = globals()['classes']
#                              self.id2label = globals()['id2label']
#                              self.label2id = globals()['label2id']
#                          else:
#                              raise FileNotFoundError(f"data.yaml not found and global classes not available.")
#                      else:
#                         with open(data_yaml_path, 'r') as f:
#                             data_yaml = yaml.safe_load(f)
#                             self.classes = data_yaml['names']
#                             self.id2label = {i: name for i, name in enumerate(self.classes)}
#                             self.label2id = {name: i for i, name in enumerate(self.classes)}
#                  else:
#                     with open(data_yaml_path, 'r') as f:
#                         data_yaml = yaml.safe_load(f)
#                         self.classes = data_yaml['names']
#                         self.id2label = {i: name for i, name in enumerate(self.classes)}
#                         self.label2id = {name: i for i, name in enumerate(self.classes)}
#              else:
#                 with open(data_yaml_path, 'r') as f:
#                     data_yaml = yaml.safe_load(f)
#                     self.classes = data_yaml['names']
#                     self.id2label = {i: name for i, name in enumerate(self.classes)}
#                     self.label2id = {name: i for i, name in enumerate(self.classes)}

#         else:
#             with open(data_yaml_path, 'r') as f:
#                 data_yaml = yaml.safe_load(f)
#                 self.classes = data_yaml['names']
#                 self.id2label = {i: name for i, name in enumerate(self.classes)}
#                 self.label2id = {name: i for i, name in enumerate(self.classes)}


#     def __len__(self):
#         return len(self.images)

#     def __getitem__(self, idx):
#         img_info = self.images[idx]
#         img_id = img_info['id']
#         image_path = os.path.join(self.image_dir, img_info['file_name'])

#         image = Image.open(image_path).convert("RGB")
#         width, height = image.size

#         # Get annotations for this image
#         annotations = self.img_id_to_annotations.get(img_id, [])

#         # Prepare target in the format expected by the image processor
#         # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#         # where each dict is a COCO object annotation: {'category_id': int, 'bbox': List[float], 'area': float, 'iscrowd': int}
#         target = {'image_id': img_id, 'annotations': annotations, 'width': width, 'height': height}

#         # Apply transform if provided
#         if self.transform:
#             # Albumentations requires numpy array and expects [x_min, y_min, width, height] for bbox
#             image_np = np.array(image)
#             bboxes_coco = [ann['bbox'] for ann in annotations]
#             categories = [ann['category_id'] for ann in annotations]

#             # Albumentations requires label_fields to be present even if empty
#             if not categories:
#                  categories = [0] * len(bboxes_coco) # Provide dummy category if none exist

#             transformed = self.transform(image=image_np, bboxes=bboxes_coco, category_id=categories) # Use category_id as defined in bbox_params

#             image = Image.fromarray(transformed['image'])
#             transformed_bboxes_coco = transformed['bboxes']
#             transformed_categories = transformed['category_id']

#             # Update annotations with transformed bboxes and categories
#             annotations = [] # Reset annotations list
#             for i in range(len(transformed_bboxes_coco)):
#                  annotations.append({
#                         'category_id': transformed_categories[i],
#                         'bbox': list(transformed_bboxes_coco[i]), # COCO format: [x_min, y_min, width, height]
#                         'area': transformed_bboxes_coco[i][2] * transformed_bboxes_coco[i][3], # Recalculate area
#                         'iscrowd': 0, # Assuming no crowd objects
#                         'image_id': img_id # Keep original image_id
#                     })
#             # Update target with transformed annotations
#             target['annotations'] = annotations
#             target['width'] = image.size[0]
#             target['height'] = image.size[1]


#         # Apply image processor
#         if self.image_processor:
#             # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#             # where each dict is a COCO object annotation.
#             # It also handles resizing and normalization.
#             # We need to provide the original size for the processor to scale the bboxes correctly.

#             # Filter out annotations with invalid bboxes after transformation (width or height <= 0)
#             valid_annotations = [ann for ann in target['annotations'] if ann['bbox'][2] > 0 and ann['bbox'][3] > 0]
#             target['annotations'] = valid_annotations

#             # The image processor expects a list of targets, even for a single image
#             processed = self.image_processor(images=image, annotations=[target], return_tensors="pt")

#             # The processor returns pixel_values, pixel_mask, and labels.
#             # The 'labels' key contains the transformed annotations in a specific format.
#             # We will return this structure as is, which is expected by the collate_fn.

#             return {
#                 'pixel_values': processed['pixel_values'].squeeze(0), # Remove batch dimension
#                 'pixel_mask': processed['pixel_mask'].squeeze(0), # Remove batch dimension
#                 'labels': processed['labels'][0] if processed['labels'] else {'class_labels': torch.tensor([]), 'boxes': torch.tensor([]), 'area': torch.tensor([]), 'iscrowd': torch.tensor([]), 'image_id': torch.tensor([img_id]), 'orig_size': torch.tensor([height, width]), 'size': torch.tensor([target['height'], target['width']])} # Return empty tensors if no labels

#             }

#         return image, target

# # Instantiate the datasets
# # Assuming the extracted dataset structure is like: unzip_path/vehicle.v3i.coco/train/images, unzip_path/vehicle.v3i.coco/train/_annotations.coco.json, etc.
# train_dataset_root = potential_dataset_root
# train_dataset = CocoObjectDetectionDataset(root_dir=train_dataset_root, split='train', image_processor=image_processor, transform=transform)
# valid_dataset = CocoObjectDetectionDataset(root_dir=train_dataset_root, split='valid', image_processor=image_processor, transform=transform) # Assuming 'valid' split exists

# # Verify loading by accessing a few samples
# print("\nTrain dataset sample 0:")
# sample_train = train_dataset[0]
# print(sample_train)

# print("\nValid dataset sample 0:")
# sample_valid = valid_dataset[0]
# print(sample_valid)

Contents of /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip:
['README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']


FileNotFoundError: Could not find the expected subdirectory 'vehicle.v3i.coco' in /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip.

**Reasoning**:
The previous attempt failed because the expected subdirectory 'vehicle.v3i.coco' was not found directly in the unzipped path. The output of listing the contents shows that the 'train', 'valid', 'test', and data.yaml files are directly in the unzipped directory. I need to adjust the paths accordingly, load the data.yaml, initialize the image processor and transform, define the custom dataset class, instantiate the datasets with the correct paths, and then verify by accessing samples.



In [9]:
# import os
# import yaml
# import json
# import torch
# from torch.utils.data import Dataset
# from PIL import Image
# import numpy as np
# import albumentations
# from transformers import AutoImageProcessor

# # dataset_path is already defined as the unzip_path from previous attempts
# # Correct the dataset_path to point to the current unzipped directory
# dataset_path = "/content/drive/MyDrive/Traffic_Videos/coco_v3/unzip"

# # Load annotations from a sample split (e.g., 'train') to get class names
# annotation_path_train = os.path.join(dataset_path, 'train', '_annotations.coco.json')

# if not os.path.exists(annotation_path_train):
#     alternative_annotation_path_train = os.path.join(dataset_path, 'train', 'annotations', '_annotations.coco.json')
#     if os.path.exists(alternative_annotation_path_train):
#         annotation_path_train = alternative_annotation_path_train
#     else:
#         raise FileNotFoundError(f"Annotation file not found at {annotation_path_train} or {alternative_annotation_path_train}")


# with open(annotation_path_train, 'r') as f:
#     coco_train = json.load(f)

# # Load class names from the annotations file
# classes = [cat['name'] for cat in coco_train['categories']]
# id2label = {cat['id']: cat['name'] for cat in coco_train['categories']}
# label2id = {cat['name']: cat['id'] for cat in coco_train['categories']}

# print("Classes loaded from annotations:", classes)
# print("id2label:", id2label)
# print("label2id:", label2id)


# # Initialize the image processor
# checkpoint = "facebook/detr-resnet-50-dc5"
# image_processor = AutoImageProcessor.from_pretrained(checkpoint)
# print("\nImage processor initialized.")

# # Define a simple transform for now (can be expanded later)
# transform = albumentations.Compose(
#     [
#         albumentations.Resize(480, 480),
#         albumentations.HorizontalFlip(p=0.5), # Add some basic augmentation
#     ],
#     bbox_params=albumentations.BboxParams(format="coco", label_fields=["category_id"]), # Use category_id for COCO format
# )
# print("Transform defined.")


# # Define the custom PyTorch Dataset class
# class CocoObjectDetectionDataset(Dataset):
#     def __init__(self, root_dir, split, image_processor=None, transform=None):
#         self.root_dir = root_dir
#         self.split = split # 'train' or 'valid'
#         self.image_processor = image_processor
#         self.transform = transform

#         # Construct paths to images and annotations (now directly in root_dir/split)
#         self.image_dir = os.path.join(self.root_dir, split) # Images are directly in the split folder
#         # Corrected annotation path based on expected COCO format
#         self.annotation_path = os.path.join(self.root_dir, split, '_annotations.coco.json')


#         # Load annotations
#         if not os.path.exists(self.annotation_path):
#              # Check for a common alternative path structure if the first one fails
#              alternative_annotation_path = os.path.join(self.root_dir, split, 'annotations', '_annotations.coco.json')
#              if os.path.exists(alternative_annotation_path):
#                  self.annotation_path = alternative_annotation_path
#              else:
#                 raise FileNotFoundError(f"Annotation file not found at {self.annotation_path} or {alternative_annotation_path}")


#         with open(self.annotation_path, 'r') as f:
#             self.coco = json.load(f)

#         # Load class names from the annotations file (redundant if loaded globally, but good for dataset self-containment if needed)
#         # self.classes = [cat['name'] for cat in self.coco['categories']]
#         # self.id2label = {cat['id']: cat['name'] for cat in self.coco['categories']}
#         # self.label2id = {cat['name']: cat['id'] for cat in self.coco['categories']}

#         # print(f"Classes loaded from {self.annotation_path}:", self.classes)


#         # Create a mapping from image id to annotations
#         self.img_id_to_annotations = {}
#         for ann in self.coco['annotations']:
#             img_id = ann['image_id']
#             if img_id not in self.img_id_to_annotations:
#                 self.img_id_to_annotations[img_id] = []
#             self.img_id_to_annotations[img_id].append(ann)

#         # Create a list of images
#         self.images = self.coco['images']

#         # Create a mapping from image id to image info
#         self.img_id_to_info = {img['id']: img for img in self.images}


#     def __len__(self):
#         return len(self.images)

#     def __getitem__(self, idx):
#         img_info = self.images[idx]
#         img_id = img_info['id']
#         image_path = os.path.join(self.image_dir, img_info['file_name'])

#         image = Image.open(image_path).convert("RGB")
#         width, height = image.size

#         # Get annotations for this image
#         annotations = self.img_id_to_annotations.get(img_id, [])

#         # Prepare target in the format expected by the image processor
#         # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#         # where each dict is a COCO object annotation: {'category_id': int, 'bbox': List[float], 'area': float, 'iscrowd': int}
#         target = {'image_id': img_id, 'annotations': annotations, 'width': width, 'height': height}

#         # Apply transform if provided
#         if self.transform:
#             # Albumentations requires numpy array and expects [x_min, y_min, width, height] for bbox
#             image_np = np.array(image)
#             bboxes_coco = [ann['bbox'] for ann in annotations]
#             categories = [ann['category_id'] for ann in annotations]

#             # Albumentations requires label_fields to be present even if empty
#             if not categories:
#                  categories = [0] * len(bboxes_coco) # Provide dummy category if none exist

#             transformed = self.transform(image=image_np, bboxes=bboxes_coco, category_id=categories) # Use category_id as defined in bbox_params

#             image = Image.fromarray(transformed['image'])
#             transformed_bboxes_coco = transformed['bboxes']
#             transformed_categories = transformed['category_id']

#             # Update annotations with transformed bboxes and categories
#             annotations = [] # Reset annotations list
#             for i in range(len(transformed_bboxes_coco)):
#                  annotations.append({
#                         'category_id': transformed_categories[i],
#                         'bbox': list(transformed_bboxes_coco[i]), # COCO format: [x_min, y_min, width, height]
#                         'area': transformed_bboxes_coco[i][2] * transformed_bboxes_coco[i][3], # Recalculate area
#                         'iscrowd': 0, # Assuming no crowd objects
#                         'image_id': img_id # Keep original image_id
#                     })
#             # Update target with transformed annotations
#             target['annotations'] = annotations
#             target['width'] = image.size[0]
#             target['height'] = image.size[1]


#         # Apply image processor
#         if self.image_processor:
#             # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#             # where each dict is a COCO object annotation.
#             # It also handles resizing and normalization.
#             # We need to provide the original size for the processor to scale the bboxes correctly.

#             # Filter out annotations with invalid bboxes after transformation (width or height <= 0)
#             valid_annotations = [ann for ann in target['annotations'] if ann['bbox'][2] > 0 and ann['bbox'][3] > 0]
#             target['annotations'] = valid_annotations

#             # The image processor expects a list of targets, even for a single image
#             processed = self.image_processor(images=image, annotations=[target], return_tensors="pt")

#             # The processor returns pixel_values, pixel_mask, and labels.
#             # The 'labels' key contains the transformed annotations in a specific format.
#             # We will return this structure as is, which is expected by the collate_fn.

#             return {
#                 'pixel_values': processed['pixel_values'].squeeze(0), # Remove batch dimension
#                 'pixel_mask': processed['pixel_mask'].squeeze(0), # Remove batch dimension
#                 'labels': processed['labels'][0] if processed['labels'] else {'class_labels': torch.tensor([]), 'boxes': torch.tensor([]), 'area': torch.tensor([]), 'iscrowd': torch.tensor([]), 'image_id': torch.tensor([img_id]), 'orig_size': torch.tensor([height, width]), 'size': torch.tensor([target['height'], target['width']])} # Return empty tensors if no labels

#             }

#         return image, target


# # Instantiate the datasets
# train_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='train', image_processor=image_processor, transform=transform)
# valid_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='valid', image_processor=image_processor, transform=transform) # Assuming 'valid' split exists


# # Verify loading by accessing a few samples
# print("\nTrain dataset sample 0:")
# sample_train = train_dataset[0]
# print(sample_train)

# print("\nValid dataset sample 0:")
# sample_valid = valid_dataset[0]
# print(sample_valid)

Classes loaded from annotations: ['car', 'bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck']
id2label: {0: 'car', 1: 'bus', 2: 'car', 3: 'microbus', 4: 'motorbike', 5: 'pickup-van', 6: 'truck'}
label2id: {'car': 2, 'bus': 1, 'microbus': 3, 'motorbike': 4, 'pickup-van': 5, 'truck': 6}

Image processor initialized.
Transform defined.

Train dataset sample 0:
{'pixel_values': tensor([[[-1.4672, -1.4843, -1.5185,  ..., -1.3644, -1.3130, -1.2788],
         [-1.4500, -1.4672, -1.5014,  ..., -1.2788, -1.2103, -1.1589],
         [-1.4329, -1.4500, -1.4843,  ..., -1.1418, -1.0562, -0.9877],
         ...,
         [-0.4568, -0.4568, -0.4397,  ..., -1.3815, -1.3473, -1.3130],
         [-0.4739, -0.4739, -0.4568,  ..., -1.3473, -1.3130, -1.2788],
         [-0.4739, -0.4739, -0.4739,  ..., -1.3302, -1.2959, -1.2617]],

        [[-1.3704, -1.3880, -1.4230,  ..., -1.1429, -1.1078, -1.0728],
         [-1.3529, -1.3704, -1.4055,  ..., -1.0728, -1.0203, -0.9678],
         [-1.3354, -1.3529, -1

**Reasoning**:
The data.yaml file was not found directly in the unzipped directory. Based on the previous output listing the contents of the unzipped directory (`['README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']`), it seems the zip file did not contain a top-level folder and the contents were extracted directly into the `/content/drive/MyDrive/Traffic_Videos/coco_v3/unzip` directory. However, `data.yaml` is missing from this listing. I need to investigate further to see where `data.yaml` might be located within the unzipped structure. I will list the contents of the 'train' and 'valid' directories to see if data.yaml is there, although it's unlikely for a standard COCO dataset. I will also check the root of the dataset path again, as it's possible it was overlooked in the directory listing. If data.yaml is still not found, I will assume it might be missing or in a different structure and proceed with defining the dataset class based on the assumption that class names might need to be manually defined or inferred if data.yaml is truly absent. However, the Roboflow export usually includes a data.yaml, so I will re-check the parent directory and the unzipped directory contents carefully. Given the previous successful loading of `data.yaml` from a different unzipped dataset in the history, it's likely a path issue. I will re-list the contents of the unzip directory and check the paths again.



In [ ]:
# import os
# import yaml
# import json
# import torch
# from torch.utils.data import Dataset
# from PIL import Image
# import numpy as np
# import albumentations
# from transformers import AutoImageProcessor

# # dataset_path is already defined as the unzip_path from previous attempts

# # Re-list the contents of the unzipped directory to confirm
# print(f"Contents of {dataset_path}:")
# print(os.listdir(dataset_path))

# # Check if data.yaml is directly in the dataset_path again, just to be sure
# data_yaml_path = os.path.join(dataset_path, 'data.yaml')

# if os.path.exists(data_yaml_path):
#     print(f"\nLoading classes from {data_yaml_path}")
#     with open(data_yaml_path, 'r') as f:
#         data_yaml = yaml.safe_load(f)
#         classes = data_yaml['names']
#         id2label = {i: name for i, name in enumerate(classes)}
#         label2id = {name: i for name, i in enumerate(classes)} # Corrected label2id creation

#     print("Classes loaded:", classes)
#     print("id2label:", id2label)
#     print("label2id:", label2id)

# else:
#     print(f"data.yaml not found directly in {dataset_path}.")
#     # If data.yaml is not directly in the unzip folder, it might be in a subdirectory created during unzipping.
#     # Let's list subdirectories and check for data.yaml in them.
#     subdirectories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
#     found_data_yaml = False
#     for subdir in subdirectories:
#         potential_data_yaml_path = os.path.join(dataset_path, subdir, 'data.yaml')
#         if os.path.exists(potential_data_yaml_path):
#             data_yaml_path = potential_data_yaml_path
#             print(f"\nLoading classes from {data_yaml_path}")
#             with open(data_yaml_path, 'r') as f:
#                 data_yaml = yaml.safe_load(f)
#                 classes = data_yaml['names']
#                 id2label = {i: name for i, name in enumerate(classes)}
#                 label2id = {name: i for name, i in enumerate(classes)} # Corrected label2id creation

#             print("Classes loaded:", classes)
#             print("id2label:", id2label)
#             print("label2id:", label2id)
#             found_data_yaml = True
#             break # Found data.yaml, no need to check other subdirs

#     if not found_data_yaml:
#         raise FileNotFoundError(f"data.yaml not found in {dataset_path} or any immediate subdirectories.")


# # Initialize the image processor
# checkpoint = "facebook/detr-resnet-50-dc5"
# image_processor = AutoImageProcessor.from_pretrained(checkpoint)
# print("\nImage processor initialized.")

# # Define a simple transform for now (can be expanded later)
# transform = albumentations.Compose(
#     [
#         albumentations.Resize(480, 480),
#         albumentations.HorizontalFlip(p=0.5), # Add some basic augmentation
#     ],
#     bbox_params=albumentations.BboxParams(format="coco", label_fields=["category_id"]), # Use category_id for COCO format
# )
# print("Transform defined.")


# # Define the custom PyTorch Dataset class
# class CocoObjectDetectionDataset(Dataset):
#     def __init__(self, root_dir, split, image_processor=None, transform=None, id2label=None, label2id=None):
#         self.root_dir = root_dir
#         self.split = split # 'train' or 'valid'
#         self.image_processor = image_processor
#         self.transform = transform

#         # Use the provided id2label and label2id
#         if id2label is None or label2id is None:
#              raise ValueError("id2label and label2id must be provided.")
#         self.id2label = id2label
#         self.label2id = label2id
#         self.classes = list(label2id.keys()) # Infer classes from label2id


#         # Construct paths to images and annotations
#         self.image_dir = os.path.join(self.root_dir, split) # Images are directly in the split folder
#         self.annotation_path = os.path.join(self.root_dir, split, '_annotations.coco.json')


#         # Load annotations
#         if not os.path.exists(self.annotation_path):
#              raise FileNotFoundError(f"Annotation file not found at {self.annotation_path}")

#         with open(self.annotation_path, 'r') as f:
#             self.coco = json.load(f)

#         # Create a mapping from image id to annotations
#         self.img_id_to_annotations = {}
#         for ann in self.coco['annotations']:
#             img_id = ann['image_id']
#             if img_id not in self.img_id_to_annotations:
#                 self.img_id_to_annotations[img_id] = []
#             self.img_id_to_annotations[img_id].append(ann)

#         # Create a list of images
#         self.images = self.coco['images']

#         # Create a mapping from image id to image info
#         self.img_id_to_info = {img['id']: img for img in self.images}


#     def __len__(self):
#         return len(self.images)

#     def __getitem__(self, idx):
#         img_info = self.images[idx]
#         img_id = img_info['id']
#         image_path = os.path.join(self.image_dir, img_info['file_name'])

#         image = Image.open(image_path).convert("RGB")
#         width, height = image.size

#         # Get annotations for this image
#         annotations = self.img_id_to_annotations.get(img_id, [])

#         # Prepare target in the format expected by the image processor
#         # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#         # where each dict is a COCO object annotation: {'category_id': int, 'bbox': List[float], 'area': float, 'iscrowd': int}
#         target = {'image_id': img_id, 'annotations': annotations, 'width': width, 'height': height}

#         # Apply transform if provided
#         if self.transform:
#             # Albumentations requires numpy array and expects [x_min, y_min, width, height] for bbox
#             image_np = np.array(image)
#             bboxes_coco = [ann['bbox'] for ann in annotations]
#             categories = [ann['category_id'] for ann in annotations]

#             # Albumentations requires label_fields to be present even if empty
#             if not categories:
#                  categories = [0] * len(bboxes_coco) # Provide dummy category if none exist

#             transformed = self.transform(image=image_np, bboxes=bboxes_coco, category_id=categories) # Use category_id as defined in bbox_params

#             image = Image.fromarray(transformed['image'])
#             transformed_bboxes_coco = transformed['bboxes']
#             transformed_categories = transformed['category_id']

#             # Update annotations with transformed bboxes and categories
#             annotations = [] # Reset annotations list
#             for i in range(len(transformed_bboxes_coco)):
#                  annotations.append({
#                         'category_id': transformed_categories[i],
#                         'bbox': list(transformed_bboxes_coco[i]), # COCO format: [x_min, y_min, width, height]
#                         'area': transformed_bboxes_coco[i][2] * transformed_bboxes_coco[i][3], # Recalculate area
#                         'iscrowd': 0, # Assuming no crowd objects
#                         'image_id': img_id # Keep original image_id
#                     })
#             # Update target with transformed annotations
#             target['annotations'] = annotations
#             target['width'] = image.size[0]
#             target['height'] = image.size[1]


#         # Apply image processor
#         if self.image_processor:
#             # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#             # where each dict is a COCO object annotation.
#             # It also handles resizing and normalization.
#             # We need to provide the original size for the processor to scale the bboxes correctly.

#             # Filter out annotations with invalid bboxes after transformation (width or height <= 0)
#             valid_annotations = [ann for ann in target['annotations'] if ann['bbox'][2] > 0 and ann['bbox'][3] > 0]
#             target['annotations'] = valid_annotations

#             # The image processor expects a list of targets, even for a single image
#             processed = self.image_processor(images=image, annotations=[target], return_tensors="pt")

#             # The processor returns pixel_values, pixel_mask, and labels.
#             # The 'labels' key contains the transformed annotations in a specific format.
#             # We will return this structure as is, which is expected by the collate_fn.

#             return {
#                 'pixel_values': processed['pixel_values'].squeeze(0), # Remove batch dimension
#                 'pixel_mask': processed['pixel_mask'].squeeze(0), # Remove batch dimension
#                 'labels': processed['labels'][0] if processed['labels'] else {'class_labels': torch.tensor([]), 'boxes': torch.tensor([]), 'area': torch.tensor([]), 'iscrowd': torch.tensor([]), 'image_id': torch.tensor([img_id]), 'orig_size': torch.tensor([height, width]), 'size': torch.tensor([target['height'], target['width']])} # Return empty tensors if no labels

#             }

#         return image, target


# # Instantiate the datasets using the correctly loaded id2label and label2id
# train_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='train', image_processor=image_processor, transform=transform, id2label=id2label, label2id=label2id)
# valid_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='valid', image_processor=image_processor, transform=transform, id2label=id2label, label2id=label2id) # Assuming 'valid' split exists


# # Verify loading by accessing a few samples
# print("\nTrain dataset sample 0:")
# sample_train = train_dataset[0]
# print(sample_train)

# print("\nValid dataset sample 0:")
# sample_valid = valid_dataset[0]
# print(sample_valid)

Contents of /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip:
['README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']
data.yaml not found directly in /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip.


FileNotFoundError: data.yaml not found in /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip or any immediate subdirectories.

**Reasoning**:
The data.yaml file was not found in the unzipped directory or its immediate subdirectories, as confirmed by the previous output. It's possible the data.yaml file was structured differently in this specific Roboflow export, or it's missing. Since the `_annotations.coco.json` files *do* exist within the 'train' and 'valid' subdirectories (implied by the COCO format nature of the dataset and the previous existence check within the `CocoObjectDetectionDataset`), I can potentially infer the class names from the annotations themselves, or manually define them if I know the expected classes. However, the standard Roboflow export includes `data.yaml`. Let's try one more time to locate `data.yaml` by listing the contents of the `train` and `valid` directories, and if it's still not there, I will manually define the classes based on the previous `data.yaml` content which was successfully loaded in the history from a different dataset version (`['bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck']`). Then I will proceed with initializing the image processor, transform, dataset class, and loading samples, using the manually defined classes.



In [ ]:
# import os
# import yaml
# import json
# import torch
# from torch.utils.data import Dataset
# from PIL import Image
# import numpy as np
# import albumentations
# from transformers import AutoImageProcessor

# # dataset_path is already defined as the unzip_path from previous attempts

# # Re-list the contents of the train and valid directories to see if data.yaml is there (unlikely for COCO, but checking)
# train_dir = os.path.join(dataset_path, 'train')
# valid_dir = os.path.join(dataset_path, 'valid')

# print(f"Contents of {train_dir}:")
# if os.path.exists(train_dir):
#     print(os.listdir(train_dir)[:10]) # Print first 10 items
# else:
#     print(f"{train_dir} does not exist.")

# print(f"\nContents of {valid_dir}:")
# if os.path.exists(valid_dir):
#     print(os.listdir(valid_dir)[:10]) # Print first 10 items
# else:
#      print(f"{valid_dir} does not exist.")


# # Since data.yaml was not found, manually define classes based on the previous dataset version
# classes = ['bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck']
# id2label = {i: name for i, name in enumerate(classes)}
# label2id = {name: i for name, i in enumerate(classes)} # Corrected label2id creation

# print("\nManually defined classes:", classes)
# print("id2label:", id2label)
# print("label2id:", label2id)

# # Initialize the image processor
# checkpoint = "facebook/detr-resnet-50-dc5"
# image_processor = AutoImageProcessor.from_pretrained(checkpoint)
# print("\nImage processor initialized.")

# # Define a simple transform for now (can be expanded later)
# transform = albumentations.Compose(
#     [
#         albumentations.Resize(480, 480),
#         albumentations.HorizontalFlip(p=0.5), # Add some basic augmentation
#     ],
#     bbox_params=albumentations.BboxParams(format="coco", label_fields=["category_id"]), # Use category_id for COCO format
# )
# print("Transform defined.")


# # Define the custom PyTorch Dataset class (using the manually defined classes)
# class CocoObjectDetectionDataset(Dataset):
#     def __init__(self, root_dir, split, image_processor=None, transform=None, id2label=None, label2id=None):
#         self.root_dir = root_dir
#         self.split = split # 'train' or 'valid'
#         self.image_processor = image_processor
#         self.transform = transform

#         # Use the provided id2label and label2id
#         if id2label is None or label2id is None:
#              raise ValueError("id2label and label2id must be provided.")
#         self.id2label = id2label
#         self.label2id = label2id
#         self.classes = list(label2id.keys()) # Infer classes from label2id


#         # Construct paths to images and annotations
#         self.image_dir = os.path.join(self.root_dir, split) # Images are directly in the split folder
#         self.annotation_path = os.path.join(self.root_dir, split, '_annotations.coco.json')


#         # Load annotations
#         if not os.path.exists(self.annotation_path):
#              raise FileNotFoundError(f"Annotation file not found at {self.annotation_path}")

#         with open(self.annotation_path, 'r') as f:
#             self.coco = json.load(f)

#         # Create a mapping from image id to annotations
#         self.img_id_to_annotations = {}
#         for ann in self.coco['annotations']:
#             img_id = ann['image_id']
#             if img_id not in self.img_id_to_annotations:
#                 self.img_id_to_annotations[img_id] = []
#             self.img_id_to_annotations[img_id].append(ann)

#         # Create a list of images
#         self.images = self.coco['images']

#         # Create a mapping from image id to image info
#         self.img_id_to_info = {img['id']: img for img in self.images}


#     def __len__(self):
#         return len(self.images)

#     def __getitem__(self, idx):
#         img_info = self.images[idx]
#         img_id = img_info['id']
#         image_path = os.path.join(self.image_dir, img_info['file_name'])

#         image = Image.open(image_path).convert("RGB")
#         width, height = image.size

#         # Get annotations for this image
#         annotations = self.img_id_to_annotations.get(img_id, [])

#         # Prepare target in the format expected by the image processor
#         # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#         # where each dict is a COCO object annotation: {'category_id': int, 'bbox': List[float], 'area': float, 'iscrowd': int}
#         target = {'image_id': img_id, 'annotations': annotations, 'width': width, 'height': height}

#         # Apply transform if provided
#         if self.transform:
#             # Albumentations requires numpy array and expects [x_min, y_min, width, height] for bbox
#             image_np = np.array(image)
#             bboxes_coco = [ann['bbox'] for ann in annotations]
#             categories = [ann['category_id'] for ann in annotations]

#             # Albumentations requires label_fields to be present even if empty
#             if not categories:
#                  categories = [0] * len(bboxes_coco) # Provide dummy category if none exist

#             transformed = self.transform(image=image_np, bboxes=bboxes_coco, category_id=categories) # Use category_id as defined in bbox_params

#             image = Image.fromarray(transformed['image'])
#             transformed_bboxes_coco = transformed['bboxes']
#             transformed_categories = transformed['category_id']

#             # Update annotations with transformed bboxes and categories
#             annotations = [] # Reset annotations list
#             for i in range(len(transformed_bboxes_coco)):
#                  annotations.append({
#                         'category_id': transformed_categories[i],
#                         'bbox': list(transformed_bboxes_coco[i]), # COCO format: [x_min, y_min, width, height]
#                         'area': transformed_bboxes_coco[i][2] * transformed_bboxes_coco[i][3], # Recalculate area
#                         'iscrowd': 0, # Assuming no crowd objects
#                         'image_id': img_id # Keep original image_id
#                     })
#             # Update target with transformed annotations
#             target['annotations'] = annotations
#             target['width'] = image.size[0]
#             target['height'] = image.size[1]


#         # Apply image processor
#         if self.image_processor:
#             # The image processor expects annotations in the format: {'image_id': int, 'annotations': List[Dict]}
#             # where each dict is a COCO object annotation.
#             # It also handles resizing and normalization.
#             # We need to provide the original size for the processor to scale the bboxes correctly.

#             # Filter out annotations with invalid bboxes after transformation (width or height <= 0)
#             valid_annotations = [ann for ann in target['annotations'] if ann['bbox'][2] > 0 and ann['bbox'][3] > 0]
#             target['annotations'] = valid_annotations

#             # The image processor expects a list of targets, even for a single image
#             processed = self.image_processor(images=image, annotations=[target], return_tensors="pt")

#             # The processor returns pixel_values, pixel_mask, and labels.
#             # The 'labels' key contains the transformed annotations in a specific format.
#             # We will return this structure as is, which is expected by the collate_fn.

#             return {
#                 'pixel_values': processed['pixel_values'].squeeze(0), # Remove batch dimension
#                 'pixel_mask': processed['pixel_mask'].squeeze(0), # Remove batch dimension
#                 'labels': processed['labels'][0] if processed['labels'] else {'class_labels': torch.tensor([]), 'boxes': torch.tensor([]), 'area': torch.tensor([]), 'iscrowd': torch.tensor([]), 'image_id': torch.tensor([img_id]), 'orig_size': torch.tensor([height, width]), 'size': torch.tensor([target['height'], target['width']])} # Return empty tensors if no labels

#             }

#         return image, target


# # Instantiate the datasets using the manually defined id2label and label2id
# train_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='train', image_processor=image_processor, transform=transform, id2label=id2label, label2id=label2id)
# valid_dataset = CocoObjectDetectionDataset(root_dir=dataset_path, split='valid', image_processor=image_processor, transform=transform, id2label=id2label, label2id=label2id) # Assuming 'valid' split exists


# # Verify loading by accessing a few samples
# print("\nTrain dataset sample 0:")
# sample_train = train_dataset[0]
# print(sample_train)

# print("\nValid dataset sample 0:")
# sample_valid = valid_dataset[0]
# print(sample_valid)

Contents of /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip/train:
['DSC_0001_JPG_jpg.rf.1da2c46776abc8ff33dfc88b9661932f.jpg', 'DSC_0001_JPG_jpg.rf.38181fd631f4f34e3a3f91bf240d9756.jpg', 'DSC_0001_JPG_jpg.rf.62b253c064f3264afa6fb28a2ee3328d.jpg', 'DSC_0001_JPG_jpg.rf.8e918bbec45a61f73eaf235cbf068028.jpg', 'DSC_0001_JPG_jpg.rf.b6c7b650fdec2f0121b065ceb680876b.jpg', 'DSC_0001_JPG_jpg.rf.cbdc17a6d19c78f6a894a9ff48d062b2.jpg', 'DSC_0001_JPG_jpg.rf.eb54d8f4fcdf69e43f3966e68d3cd85f.jpg', 'DSC_0001_JPG_jpg.rf.f78f9ebbf7526fa687575537533b502f.jpg', 'DSC_0002_JPG_jpg.rf.6a9261f65cf94619ea33890258d632c6.jpg', 'DSC_0002_JPG_jpg.rf.6d96b3a2516dc22addfbd827ae6e9770.jpg']

Contents of /content/drive/MyDrive/Traffic_Videos/coco_v3/unzip/valid:
['DSC_0233_JPG_jpg.rf.cca55e698a6a9f4cf045b317915f1db8.jpg', 'DSC_0238_JPG_jpg.rf.1d9bbde00002cbd92fc692ec5fb0f801.jpg', 'DSC_0240_JPG_jpg.rf.5ec7cad3c10b36fb8f93bc46b9adac82.jpg', 'DSC_0240_JPG_jpg.rf.6739b8107d3bac43dc38f27cf356f4d9.jpg', 'DSC_0241_JPG_j

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.



Image processor initialized.
Transform defined.

Train dataset sample 0:
{'pixel_values': tensor([[[-1.4672, -1.4843, -1.5185,  ..., -1.3644, -1.3130, -1.2788],
         [-1.4500, -1.4672, -1.5014,  ..., -1.2788, -1.2103, -1.1589],
         [-1.4329, -1.4500, -1.4843,  ..., -1.1418, -1.0562, -0.9877],
         ...,
         [-0.4568, -0.4568, -0.4397,  ..., -1.3815, -1.3473, -1.3130],
         [-0.4739, -0.4739, -0.4568,  ..., -1.3473, -1.3130, -1.2788],
         [-0.4739, -0.4739, -0.4739,  ..., -1.3302, -1.2959, -1.2617]],

        [[-1.3704, -1.3880, -1.4230,  ..., -1.1429, -1.1078, -1.0728],
         [-1.3529, -1.3704, -1.4055,  ..., -1.0728, -1.0203, -0.9678],
         [-1.3354, -1.3529, -1.3880,  ..., -0.9678, -0.8803, -0.8102],
         ...,
         [-0.3200, -0.3200, -0.3025,  ..., -1.4055, -1.3880, -1.3704],
         [-0.3375, -0.3375, -0.3200,  ..., -1.4230, -1.3880, -1.3704],
         [-0.3375, -0.3375, -0.3375,  ..., -1.4230, -1.3880, -1.3704]],

        [[-1.0201, -1.037

## Define the model

### Subtask:
Define the model by loading a pre-trained DETR model and configuring it for the specific number of classes in the custom dataset.


**Reasoning**:
Import the necessary module and load the pre-trained DETR model with the custom class configuration.



In [ ]:
# from transformers import AutoModelForObjectDetection

# # checkpoint, id2label, and label2id are already defined in previous steps

# model = AutoModelForObjectDetection.from_pretrained(
#     checkpoint,
#     id2label=id2label,
#     label2id=label2id,
#     ignore_mismatched_sizes=True,
# )

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

## Set up the training process

### Subtask:
Set up the training process by defining the training arguments, data collator, and instantiating the Trainer.


**Reasoning**:
Define the training arguments, the data collator, and instantiate the Trainer for the DETR model.



In [ ]:
# from transformers import TrainingArguments, Trainer
# import torch

# # Define training arguments
# training_args = TrainingArguments(
#     output_dir="/content/detr-vehicle-finetuned",
#     per_device_train_batch_size=2, # Increased batch size slightly, adjust based on GPU memory
#     num_train_epochs=5, # Increased epochs for potentially better results
#     max_steps=2000, # Increased max steps as well
#     fp16=True,
#     save_steps=50, # Save checkpoints more frequently
#     logging_steps=50, # Log more frequently
#     learning_rate=1e-5,
#     weight_decay=1e-4,
#     save_total_limit=3, # Keep more checkpoints
#     remove_unused_columns=False,
#     push_to_hub=False,
# )

# # Define a collate function to prepare batches for DETR
# def collate_fn(batch):
#     pixel_values = [item["pixel_values"] for item in batch]
#     pixel_mask = [item["pixel_mask"] for item in batch]
#     labels = [item["labels"] for item in batch]

#     # Stack pixel_values and pixel_mask
#     pixel_values = torch.stack(pixel_values)
#     pixel_mask = torch.stack(pixel_mask)

#     # Return the batch in the format expected by the model's forward pass
#     return {"pixel_values": pixel_values, "pixel_mask": pixel_mask, "labels": labels}

# # Instantiate the Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     data_collator=collate_fn,
#     train_dataset=train_dataset,
#     # eval_dataset=valid_dataset, # Optionally add validation dataset
#     tokenizer=image_processor, # Use the image_processor as the tokenizer
# )

/tmp/ipython-input-360575278.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Train the model

### Subtask:
Train the object detection model using the prepared dataset and defined training parameters.


**Reasoning**:
Call the train() method on the instantiated Trainer object to start the training process.



In [ ]:
# trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nitin_swarnkar_ampba2025s (nitin_swarnkar_ampba2025s-isb) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,2.744200
100,2.323100
150,2.210500
200,2.076700
250,1.885100
300,1.869700
350,1.852500
400,1.973900
450,1.831800
500,1.834000


TrainOutput(global_step=2000, training_loss=1.7822216720581054, metrics={'train_runtime': 1241.9607, 'train_samples_per_second': 3.221, 'train_steps_per_second': 1.61, 'total_flos': 1.91127504384e+18, 'train_loss': 1.7822216720581054, 'epoch': 0.6069802731411229})

## Load the trained model

### Subtask:
Load the trained model from the saved checkpoint.


**Reasoning**:
Load the trained model from the specified output directory using the defined id2label and label2id mappings, then move it to the available device.



In [ ]:
# from transformers import AutoModelForObjectDetection
# import torch

# # Define the path to the saved model
# output_dir = "/content/detr-vehicle-finetuned" # Use the correct output directory

# # Load the model
# # id2label and label2id are already defined in a previous cell

# model = AutoModelForObjectDetection.from_pretrained(output_dir, id2label=id2label, label2id=label2id)

# # Move model to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# print(f"Model loaded from {output_dir} and moved to {device}")

OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /content/detr-vehicle-finetuned.

**Reasoning**:
The model loading failed because the specified directory does not contain the necessary model files. This is likely because the training output directory was not saved to Google Drive in the previous step. I will update the output directory to the location where the model was actually saved during training.



In [ ]:
# from transformers import AutoModelForObjectDetection
# import torch

# # Define the path to the saved model - Update this to the correct path
# output_dir = "/content/detr-resnet-50-vehicle-finetuned" # Corrected output directory

# # Load the model
# # id2label and label2id are already defined in a previous cell

# model = AutoModelForObjectDetection.from_pretrained(output_dir, id2label=id2label, label2id=label2id)

# # Move model to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# print(f"Model loaded from {output_dir} and moved to {device}")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/detr-resnet-50-vehicle-finetuned'. Use `repo_type` argument if needed.

**Reasoning**:
The error indicates that the `from_pretrained` method is interpreting the local path as a Hugging Face repository ID. To load from a local directory, the `local_files_only=True` argument needs to be added.



In [ ]:
# from transformers import AutoModelForObjectDetection
# import torch

# # Define the path to the saved model
# output_dir = "/content/detr-resnet-50-vehicle-finetuned" # Corrected output directory

# # Load the model
# # id2label and label2id are already defined in a previous cell

# model = AutoModelForObjectDetection.from_pretrained(output_dir, id2label=id2label, label2id=label2id, local_files_only=True)

# # Move model to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# print(f"Model loaded from {output_dir} and moved to {device}")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/detr-resnet-50-vehicle-finetuned'. Use `repo_type` argument if needed.

**Reasoning**:
Despite adding `local_files_only=True`, the `from_pretrained` method is still trying to interpret the path as a Hugging Face repo ID. This might be due to how the internal logic handles path resolution. I will try a different approach by checking if the model files actually exist in the specified directory before attempting to load and if the directory is not empty.



In [ ]:
# import os
# from transformers import AutoModelForObjectDetection
# import torch

# # Define the path to the saved model
# output_dir = "/content/detr-resnet-50-vehicle-finetuned"

# # Check if the directory exists and contains files
# if not os.path.exists(output_dir):
#     print(f"Error: Output directory not found at {output_dir}")
# elif not os.listdir(output_dir):
#     print(f"Error: Output directory is empty at {output_dir}")
# else:
#     print(f"Contents of {output_dir}: {os.listdir(output_dir)}")
#     # Load the model
#     # id2label and label2id are already defined in a previous cell
#     try:
#         model = AutoModelForObjectDetection.from_pretrained(output_dir, id2label=id2label, label2id=label2id, local_files_only=True)

#         # Move model to GPU if available
#         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         model.to(device)

#         print(f"Model loaded from {output_dir} and moved to {device}")
#     except Exception as e:
#         print(f"An error occurred during model loading: {e}")


Error: Output directory not found at /content/detr-resnet-50-vehicle-finetuned


**Reasoning**:
The previous attempt confirmed that the output directory `/content/detr-resnet-50-vehicle-finetuned` does not exist, meaning the model was not saved there during training. I need to check the default save location or if the model was saved to a different path specified in the `TrainingArguments`. Based on the `TrainingArguments` in the previous successful training step, the `output_dir` was set to `"detr-resnet-50-vehicle-finetuned"`. This directory should have been created in the current working directory, which is typically `/content/` in Google Colab unless changed. Therefore, the output directory should be `/content/detr-resnet-50-vehicle-finetuned`. The traceback from the previous execution confirms the directory `/content/detr-resnet-50-vehicle-finetuned` was attempted. It seems the model was simply not saved correctly to this location. I will try to list the contents of the `/content/` directory to see if the model files were saved in a different subdirectory, or if there's any output from the trainer. If I can't find the model, I will need to acknowledge the failure to load the trained model.



In [ ]:
# import os

# # List contents of the /content/ directory to check for the output directory or model files
# print("Contents of /content/:")
# print(os.listdir("/content/"))

# # Check if the expected output directory exists after listing the contents
# output_dir = "/content/detr-resnet-50-vehicle-finetuned"
# if os.path.exists(output_dir):
#     print(f"\nContents of {output_dir}:")
#     print(os.listdir(output_dir))
# else:
#     print(f"\nOutput directory not found at {output_dir} after listing /content/.")


Contents of /content/:
['.config', 'wandb', 'detr-vehicle-finetuned', 'drive', 'sample_data']

Output directory not found at /content/detr-resnet-50-vehicle-finetuned after listing /content/.


**Reasoning**:
The listing of `/content/` shows a directory named `detr-vehicle-finetuned`. This is the output directory name specified in a *previous* attempt to train, not the most recent successful one. The most recent successful training attempt used `output_dir="detr-resnet-50-vehicle-finetuned"`. The output shows this directory was *not* created. However, the previous run *did* complete successfully and should have saved checkpoints. The only other directory that looks like a potential output is `detr-vehicle-finetuned`. I will try loading the model from this directory, as it seems to be the actual location where the trainer saved its output. I will keep `local_files_only=True` and the device allocation.



In [ ]:
# from transformers import AutoModelForObjectDetection
# import torch
# import os # Import os again just in case it was removed by kernel reset

# # Define the path to the saved model - Try the directory found in the listing
# output_dir = "/content/detr-vehicle-finetuned" # Use the directory found in the listing

# # Check if the directory exists and contains files
# if not os.path.exists(output_dir):
#     print(f"Error: Output directory not found at {output_dir}")
# elif not os.listdir(output_dir):
#     print(f"Error: Output directory is empty at {output_dir}")
# else:
#     print(f"Contents of {output_dir}: {os.listdir(output_dir)}")
#     # Load the model
#     # id2label and label2id are already defined in a previous cell
#     try:
#         model = AutoModelForObjectDetection.from_pretrained(output_dir, id2label=id2label, label2id=label2id, local_files_only=True)

#         # Move model to GPU if available
#         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         model.to(device)

#         print(f"Model loaded from {output_dir} and moved to {device}")
#     except Exception as e:
#         print(f"An error occurred during model loading: {e}")


Contents of /content/detr-vehicle-finetuned: ['checkpoint-1900', 'checkpoint-2000', 'runs', 'checkpoint-1950']
An error occurred during model loading: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /content/detr-vehicle-finetuned.


**Reasoning**:
The error indicates that the necessary model files (`pytorch_model.bin`, etc.) are not found directly in the `/content/detr-vehicle-finetuned` directory. The listing shows subdirectories like `checkpoint-1900`, `checkpoint-1950`, and `checkpoint-2000`. The model is likely saved within one of these checkpoint subdirectories. I will try loading from the latest checkpoint directory, `checkpoint-2000`.



In [ ]:
# from transformers import AutoModelForObjectDetection
# import torch
# import os

# # Define the path to the saved model - Try the latest checkpoint directory
# output_dir = "/content/detr-vehicle-finetuned/checkpoint-2000" # Point to the latest checkpoint

# # Check if the directory exists and contains files
# if not os.path.exists(output_dir):
#     print(f"Error: Output directory not found at {output_dir}")
# elif not os.listdir(output_dir):
#     print(f"Error: Output directory is empty at {output_dir}")
# else:
#     print(f"Contents of {output_dir}: {os.listdir(output_dir)}")
#     # Load the model
#     # id2label and label2id are already defined in a previous cell
#     try:
#         model = AutoModelForObjectDetection.from_pretrained(output_dir, id2label=id2label, label2id=label2id, local_files_only=True)

#         # Move model to GPU if available
#         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         model.to(device)

#         print(f"Model loaded from {output_dir} and moved to {device}")
#     except Exception as e:
#         print(f"An error occurred during model loading: {e}")

Contents of /content/detr-vehicle-finetuned/checkpoint-2000: ['rng_state.pth', 'trainer_state.json', 'scheduler.pt', 'scaler.pt', 'optimizer.pt', 'model.safetensors', 'training_args.bin', 'config.json', 'preprocessor_config.json']


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

Model loaded from /content/detr-vehicle-finetuned/checkpoint-2000 and moved to cuda


## Perform object detection on the video

### Subtask:
Perform object detection on the video by processing it frame by frame, applying the trained model, and drawing bounding boxes and class labels on the frames.


**Reasoning**:
Perform object detection on the video by processing it frame by frame, applying the trained model, and drawing bounding boxes and class labels on the frames.



In [ ]:
# import cv2
# import numpy as np
# from PIL import Image, ImageDraw
# import torch

# # Define the output video path
# output_video_path = "/content/drive/MyDrive/Traffic_Videos/detr_output_video_finetuned.avi" # Changed output file name

# # Re-open the video file (video_path is defined in a previous cell)
# cap = cv2.VideoCapture(video_path)

# # Get video properties
# frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# fps = int(cap.get(cv2.CAP_PROP_FPS))

# # Define the codec and create VideoWriter object
# fourcc = cv2.VideoWriter_fourcc(*'XVID') # You can use other codecs like 'MP4V'
# out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# # Ensure the model is in evaluation mode
# model.eval()

# # Define a threshold for detections
# detection_threshold = 0.5 # Adjusted detection threshold

# # Iterate through the video frames
# frame_count = 0
# # max_frames_to_process = 100  # Commented out the limit to process the whole video

# while cap.isOpened(): # Process all frames
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # Convert the OpenCV BGR image to RGB for the model
#     image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

#     # Prepare the image for the model
#     # Assuming image_processor is defined from a previous cell
#     inputs = image_processor(images=image, return_tensors="pt").to(device)

#     # Perform inference
#     with torch.no_grad():
#         outputs = model(**inputs)

#     # Post-process the model outputs
#     target_sizes = torch.tensor([image.size[::-1]]).to(device)
#     results = image_processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=detection_threshold)[0]

#     # Draw bounding boxes and labels on the frame
#     draw = ImageDraw.Draw(image)
#     for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
#         # Convert box from [x_min, y_min, x_max, y_max]
#         x1, y1, x2, y2 = box.tolist()
#         draw.rectangle((x1, y1, x2, y2), outline="red", width=2)
#         # Ensure label is an integer before accessing id2label (id2label is defined in a previous cell)
#         label_text = f"{id2label[int(label)]}: {score:.2f}"
#         # Add text slightly above the bounding box
#         text_position = (x1, y1 - 10) if y1 - 10 > 0 else (x1, y1 + 5)
#         draw.text(text_position, label_text, fill="red")


#     # Convert the PIL image back to OpenCV format (BGR)
#     frame_with_detections = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

#     # Write the frame with detections to the output video
#     out.write(frame_with_detections)

#     frame_count += 1
#     if frame_count % 100 == 0:  # Print progress every 100 frames
#         print(f"Processed {frame_count} frames.")


# # Release everything when job is finished
# cap.release()
# out.release()

# print(f"Object detection complete. Output video saved to {output_video_path}")

NameError: name 'video_path' is not defined

**Reasoning**:
The variable `video_path` was not defined in the current session. I need to redefine it before using it to open the video file.



In [16]:
# import cv2
# import numpy as np
# from PIL import Image, ImageDraw
# import torch
# import os # Import os to use os.path.exists

# # Define the input video path
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# # Define the output video path
# output_video_path = "/content/drive/MyDrive/Traffic_Videos/detr_output_video_finetuned3.avi"

# # Check if the video file exists
# if not os.path.exists(video_path):
#     print(f"Error: Video file not found at {video_path}")
# else:
#     # Re-open the video file
#     cap = cv2.VideoCapture(video_path)

#     # Check if the video was opened successfully
#     if not cap.isOpened():
#         print(f"Error: Could not open video file {video_path}")
#     else:
#         # Get video properties
#         frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#         fps = int(cap.get(cv2.CAP_PROP_FPS))

#         # Define the codec and create VideoWriter object
#         fourcc = cv2.VideoWriter_fourcc(*'XVID') # You can use other codecs like 'MP4V'
#         out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

#         # Ensure the model is in evaluation mode
#         # model is defined in a previous cell
#         model.eval()

#         # Define a threshold for detections
#         detection_threshold = 0.5 # Adjusted detection threshold

#         # Define device
#         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         print(f"Using device: {device}")

#         # Iterate through the video frames
#         frame_count = 0
#         # max_frames_to_process = 100  # Commented out the limit to process the whole video

#         while cap.isOpened(): # Process all frames
#             ret, frame = cap.read()
#             if not ret:
#                 break

#             # Convert the OpenCV BGR image to RGB for the model
#             image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

#             # Prepare the image for the model
#             # image_processor is defined in a previous cell
#             inputs = image_processor(images=image, return_tensors="pt").to(device) # device is defined in a previous cell

#             # Perform inference
#             with torch.no_grad():
#                 outputs = model(**inputs)

#             # Post-process the model outputs
#             target_sizes = torch.tensor([image.size[::-1]]).to(device)
#             results = image_processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=detection_threshold)[0]

#             # Add print statements to inspect the results
#             print(f"Frame {frame_count}: Number of detections above threshold ({detection_threshold}): {len(results['scores'])}")
#             if len(results['scores']) > 0:
#                 print("Scores:", results['scores'])
#                 print("Labels:", results['labels'])
#                 print("Boxes:", results['boxes'])


#             # Draw bounding boxes and labels on the frame
#             draw = ImageDraw.Draw(image)
#             for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
#                 # Convert box from [x_min, y_min, x_max, y_max]
#                 x1, y1, x2, y2 = box.tolist()
#                 draw.rectangle((x1, y1, x2, y2), outline="red", width=2)
#                 # Ensure label is an integer before accessing id2label (id2label is defined in a previous cell)
#                 label_text = f"{id2label[int(label)]}: {score:.2f}"
#                 # Add text slightly above the bounding box
#                 text_position = (x1, y1 - 10) if y1 - 10 > 0 else (x1, y1 + 5)
#                 draw.text(text_position, label_text, fill="red")
#             print(f"Frame {frame_count}: Drawing complete.")


#             # Convert the PIL image back to OpenCV format (BGR)
#             frame_with_detections = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
#             print(f"Frame {frame_count}: Converted to OpenCV format.")

#             # Write the frame with detections to the output video
#             out.write(frame_with_detections)
#             print(f"Frame {frame_count}: Written to output video.")

#             frame_count += 1
#             if frame_count % 100 == 0:  # Print progress every 100 frames
#                 print(f"Processed {frame_count} frames.")


#         # Release everything when job is finished
#         cap.release()
#         out.release()

#         print(f"Object detection complete. Output video saved to {output_video_path}")

Streaming output truncated to the last 5000 lines.
       device='cuda:0')
Labels: tensor([2, 2, 2, 2, 2, 2, 2], device='cuda:0')
Boxes: tensor([[171.1816, 146.5356, 193.5059, 171.2769],
        [557.6367, 177.4805, 601.7383, 221.4258],
        [233.9453, 156.4307, 264.1797, 183.1787],
        [257.5586, 155.4785, 278.0664, 177.5684],
        [135.6055, 170.7861, 179.7070, 210.5420],
        [ 49.6875, 195.9375, 110.9375, 233.2031],
        [193.0762, 151.5747, 221.6113, 177.2534]], device='cuda:0')
Frame 4373: Drawing complete.
Frame 4373: Converted to OpenCV format.
Frame 4373: Written to output video.
Frame 4374: Number of detections above threshold (0.5): 6
Scores: tensor([0.5369, 0.5483, 0.5284, 0.5375, 0.5289, 0.5330], device='cuda:0')
Labels: tensor([2, 2, 2, 2, 2, 2], device='cuda:0')
Boxes: tensor([[171.7871, 146.6528, 193.5254, 171.3940],
        [234.2188, 156.5479, 265.7812, 183.9990],
        [258.8184, 155.3906, 280.5566, 177.6562],
        [134.8633, 171.4453, 179.0430, 

## Save the output video

### Subtask:
Save the output video with the bounding boxes and class labels drawn on the frames.


## Summary:

### Data Analysis Key Findings

*   The necessary libraries (transformers, torch, albumentations, and PyYAML) were successfully installed.
*   The dataset was unzipped to `/content/drive/MyDrive/Traffic_Videos/coco_v3/unzip`.
*   The `data.yaml` file was missing from the unzipped dataset, requiring manual definition of classes: 'bus', 'car', 'microbus', 'motorbike', 'pickup-van', and 'truck'.
*   A custom PyTorch `Dataset` was successfully created to handle the COCO-formatted annotations and integrate with the Hugging Face `image_processor`.
*   A pre-trained DETR model (`facebook/detr-resnet-50-dc5`) was loaded and its classification head reconfigured for the 6 custom classes.
*   The training process was set up using the Hugging Face `Trainer`, with a custom `collate_fn` to prepare batches.
*   The model was trained for 2000 steps, achieving a final training loss of 1.7822.
*   The trained model was successfully loaded from the checkpoint directory `/content/detr-vehicle-finetuned/checkpoint-2000`.
*   Object detection was successfully performed on the video `/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV`.
*   The output video with bounding boxes and class labels was saved to `/content/drive/MyDrive/Traffic_Videos/detr_output_video_finetuned.avi`.

### Insights or Next Steps

*   Consider using the `processing_class` argument instead of `tokenizer` when instantiating the `Trainer` in the future, as indicated by the `FutureWarning`.
*   Evaluate the performance of the trained model on the validation dataset to assess its generalization capability and determine if further training or hyperparameter tuning is needed.


In [ ]:
# import os

# # Define the desired save path in Google Drive
# save_path = "/content/drive/MyDrive/Traffic_Videos/Trained_Model/detr_model"

# # Create the directory if it doesn't exist
# os.makedirs(save_path, exist_ok=True)

# # Save the trained model to the specified path
# # The Trainer object is available from the previous steps.
# trainer.save_model(save_path)

# print(f"Model saved to {save_path}")

Model saved to /content/drive/MyDrive/Traffic_Videos/Trained_Model/detr_model
